# DistilBERT financial sentiment

This notebook evaluates a smaller pretrained checkpoint first and optionally fine-tunes it.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.text_utils import load_data, metrics_table, set_seed, summarize_splits
from utils.transformer_utils import build_transformer_trainer, evaluate_transformer_checkpoint

set_seed(42)

I0000 00:00:1777752279.578651  261190 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752279.608649  261190 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777752280.369342  261190 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777752280.370024  261190 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will 

In [2]:
MODEL_NAME = "AnkitAI/distilbert-base-uncased-financial-news-sentiment-analysis"
MODEL_LABEL_ORDER = ["negative", "neutral", "positive"]
df = load_data()
display(summarize_splits(df))
MODEL_NAME

,rows,mean_words,negative,neutral,positive
split,,,,,
test,23566,17.30,5817,7306,10443
train,77589,18.01,18856,25403,33330
validation,23567,17.36,5722,7261,10584


'AnkitAI/distilbert-base-uncased-financial-news-sentiment-analysis'

In [3]:
results = evaluate_transformer_checkpoint(MODEL_NAME, MODEL_LABEL_ORDER, df, split="test", max_length=128, batch_size=32)
display(metrics_table(results))
display(results["report"])
display(results["confusion_matrix"])
results["predictions"].head()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

evaluating AnkitAI/distilbert-base-uncased-financial-news-sentiment-analysis:   0%|          | 0/737 [00:00<?,…

,metric,value
0,accuracy,0.583340
1,macro_precision,0.644347
2,macro_recall,0.608950
3,macro_f1,0.591637
4,weighted_f1,0.583167


,precision,recall,f1-score,support
negative,0.679918,0.626268,0.651991,5817.00000
neutral,0.445843,0.775801,0.566262,7306.00000
positive,0.807279,0.424782,0.556657,10443.00000
accuracy,0.583340,0.583340,0.583340,0.58334
macro avg,0.644347,0.608950,0.591637,23566.00000
weighted avg,0.663788,0.583340,0.583167,23566.00000


,negative,neutral,positive
negative,3643,1772,402
neutral,981,5668,657
positive,734,5273,4436


,text,label,prediction
0,hqge ltnc hbrm enzc eeenf halb azfl maxd mmex ...,positive,neutral
1,econx november nonfarm private payrolls k vs k...,positive,neutral
2,regulatory news the nomination committee of cy...,neutral,neutral
3,amazon labor union is now seeking to represent...,positive,neutral
4,greene king s third quarter sales boosted by f...,positive,positive


In [4]:
trainer, datasets, tokenizer = build_transformer_trainer(
    df,
    model_name=MODEL_NAME,
    model_label_order=MODEL_LABEL_ORDER,
    output_dir=str(ROOT / "artifacts" / "distilbert_financial"),
    max_length=128,
    epochs=2,
    train_batch_size=16,
    eval_batch_size=32,
)

trainer.train()
trainer.evaluate(datasets["test"])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/77589 [00:00<?, ? examples/s]

Map:   0%|          | 0/23567 [00:00<?, ? examples/s]

Map:   0%|          | 0/23566 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
1,0.568962,0.337796,0.873425,0.869670,0.867351,0.868477,0.873293
2,0.281086,0.335031,0.890101,0.887310,0.883803,0.885436,0.889926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0.281086,0.330904,2,0.892218,0.890282,0.886985,0.888473,0.892065


{'eval_loss': 0.3309042751789093,
 'eval_accuracy': 0.8922176016294662,
 'eval_macro_precision': 0.890281684710489,
 'eval_macro_recall': 0.8869847827512313,
 'eval_macro_f1': 0.8884727759488413,
 'eval_weighted_f1': 0.8920650614772482}